In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import json
import pandas as pd
import numpy as np
import os
import re
import glob
import joblib
import random
import string
from collections import Counter
from urllib.parse import urlparse
from tqdm import tqdm

from dotenv import load_dotenv
load_dotenv()

import sys
sys.path.append("../..")

from findai.entities import ThreadedConversation, Message

## Anonymization functions

In [ ]:
def generate_random_str_id(length=8, existing_ids=set()):
    while True:
        random_id = ''.join(random.choices(string.ascii_letters + string.digits, k=length))
        if random_id not in existing_ids:
            return random_id


def anonymize_user_id(user_id, user_id_mapping, ids_used):
    user_id = str(user_id)
    if user_id not in user_id_mapping:
        anonymized_id = generate_random_str_id(existing_ids=ids_used)
        user_id_mapping[user_id] = anonymized_id
        ids_used.add(anonymized_id)
    return user_id_mapping[user_id]


def process_reddit_text(text, user_id_mapping, ids_used):
    if not isinstance(text, str):
        return str(text) if text is not None else ""

    # 1. Truncate URLs to domain only
    def replace_url(match):
        url = match.group(0)
        trailing = ''
        while url and url[-1] in ')]}.,!?':
            trailing = url[-1] + trailing
            url = url[:-1]
        try:
            parsed = urlparse(url)
            replaced = f"{parsed.scheme}://{parsed.netloc}" if parsed.netloc else "[LINK]"
        except Exception:
            replaced = "[LINK]"
        return replaced + trailing
    text = re.sub(r'https?://[^\s]+', replace_url, text)

    # 2. Replace email addresses
    text = re.sub(r'\b[\w.+-]+@[\w.-]+\.[a-zA-Z]{2,}\b', '[EMAIL]', text)

    # 3. Anonymize /u/username mentions (standalone only)
    def replace_u_mention(match):
        username = match.group(1)
        anon = anonymize_user_id(username, user_id_mapping, ids_used)
        return f"/u/{anon}"
    text = re.sub(r'(?<!\S)/u/([\w-]+)', replace_u_mention, text)

    # 4. Anonymize @username mentions
    def replace_mention(match):
        username = match.group(1)
        anon = anonymize_user_id(username, user_id_mapping, ids_used)
        return f"@{anon}"
    text = re.sub(r'(?<!\S)@([\w.-]+)', replace_mention, text)

    return text

## Process, anonymize, and save by channel

In [ ]:
reddit_path = "../../data/reddit/01_raw_data"
data_reddit = sorted(glob.glob(os.path.join(reddit_path, "*.joblib")))
os.makedirs("../../data/reddit/02_processed_data", exist_ok=True)

all_stats_reddit = []
all_threads_filtered = {}

for data_file in tqdm(data_reddit):
    filename = os.path.basename(data_file).replace(".joblib", "")
    parts = filename.split("-", 2)
    lang, channel_name = parts[1], parts[2]
    key = f"{lang}-{channel_name}"

    all_conversations = joblib.load(data_file)
    print(f"\n{key}: {len(all_conversations)} threads loaded")

    user_id_mapping = {}
    ids_used = set()
    thread_id_mapping = {}
    thread_message_mappings = {}

    for thread_idx, conv in enumerate(all_conversations):
        anon_thread_id = f"thread_{thread_idx}"
        if conv.id is not None:
            thread_id_mapping[str(conv.id)] = anon_thread_id

        local_msg_mapping = {}
        for msg_idx, message in enumerate(conv.previous_comments):
            if str(message.user_id) == "[deleted]":
                continue
            if message.id is not None:
                local_msg_mapping[str(message.id)] = f"{anon_thread_id}_msg_{msg_idx}"
            anonymize_user_id(str(message.user_id), user_id_mapping, ids_used)
        thread_message_mappings[thread_idx] = local_msg_mapping

    anon_conversations = []
    for thread_idx, conv in enumerate(tqdm(all_conversations, leave=False)):
        anon_thread_id = thread_id_mapping.get(str(conv.id)) if conv.id is not None else None
        local_msg_mapping = thread_message_mappings[thread_idx]

        anon_initial_post = (
            process_reddit_text(conv.initial_post, user_id_mapping, ids_used)
            if conv.initial_post is not None else None
        )
        anon_messages = [
            Message(
                id=local_msg_mapping.get(str(message.id)) if message.id is not None else None,
                message_text=process_reddit_text(message.message_text, user_id_mapping, ids_used),
                user_id=anonymize_user_id(str(message.user_id), user_id_mapping, ids_used),
                answer_to_message_id=local_msg_mapping.get(str(message.answer_to_message_id)) if message.answer_to_message_id is not None else None,
                timestamp=message.timestamp,
                metadata=message.metadata,
            )
            for message in conv.previous_comments
            if str(message.user_id) != "[deleted]"
        ]
        if anon_messages:
            anon_conversations.append(ThreadedConversation(
                id=anon_thread_id,
                initial_post=anon_initial_post,
                previous_comments=anon_messages,
            ))

    del all_conversations
    del thread_message_mappings

    all_text = " ".join(
        message.message_text
        for conv in anon_conversations
        for message in conv.previous_comments
    )
    original_ids = set(user_id_mapping.keys())
    found_at = set(re.findall(r'(?<!\S)@([\w.-]+)', all_text))
    found_u = set(re.findall(r'(?<!\S)/u/([\w-]+)', all_text))
    leaked = (found_at | found_u) & original_ids
    leaked = set(s for s in leaked if len(s) > 3)
    del all_text
    print(f"  Leaked IDs: {leaked or 'none'}")

    shown = set()
    ids_used_cur = set(user_id_mapping.values())
    for conv in anon_conversations:
        for message in conv.previous_comments:
            for uid in leaked:
                if uid not in shown and (f"@{uid}" in message.message_text or f"/u/{uid}" in message.message_text):
                    print(f"    {uid} → {message.message_text[:150]}")
                    shown.add(uid)
                if f"@{uid}" in message.message_text:
                    anon = anonymize_user_id(uid, user_id_mapping, ids_used_cur)
                    message.message_text = message.message_text.replace(f"@{uid}", f"@{anon}")
                if f"/u/{uid}" in message.message_text:
                    anon = anonymize_user_id(uid, user_id_mapping, ids_used_cur)
                    message.message_text = message.message_text.replace(f"/u/{uid}", f"/u/{anon}")

    user_thread_counter = Counter()
    for conv in anon_conversations:
        for uid in set(m.user_id for m in conv.previous_comments):
            user_thread_counter[uid] += 1
    eligible_users = set(uid for uid, count in user_thread_counter.items() if count >= 10)
    filtered = [
        conv for conv in anon_conversations
        if any(m.user_id in eligible_users for m in conv.previous_comments)
    ]
    print(f"  Eligible users: {len(eligible_users)}, filtered threads: {len(filtered)}")

    all_stats_reddit.append({
        'language': lang,
        'channel_name': channel_name,
        'total_threads': len(anon_conversations),
        'total_users': len(user_id_mapping),
        'eligible_users': len(eligible_users),
        'filtered_threads': len(filtered),
        'leaked_ids': len(leaked),
    })

    joblib.dump(anon_conversations, f"../../data/reddit/02_processed_data/{key}_all.joblib")
    del anon_conversations